# WiFi fingerprinting: UJIIndoorLoc và các benchmark trên Google Colab

Notebook lấy mã nền từ `main`, sau đó tự bổ sung plugin mới từ tất cả nhánh `feature/*` mà token đọc được. Menu phát hiện model, các biến thể KAN gốc, FastKAN, BSRBF-KAN, S-KAN, FC-KAN và tùy chọn Mixer cho từng biến thể, cùng các model KNN/WKNN, RSS powered, CDAE-CNN và SAE/CDAE-MMJ từ các nhánh đã tải. File đã có ở `main` không bị ghi đè; thay đổi file dùng chung cần được merge vào `main`. **Không lưu GitHub token trong notebook.** Bật GPU trước khi chạy model neural nặng. Chọn UJIIndoorLoc hoặc HCXY, SYL, TUJI1, HUS, DLULIB từ ZIP. Official test chỉ dùng để báo cáo sau khi chọn model bằng internal validation.


In [ ]:
import io
import json
import shutil
import subprocess
import sys
import urllib.error
import urllib.request
import zipfile
from pathlib import Path
from google.colab import files, userdata

REPO = "VietUniverse/ips-wifi-research"
API = f"https://api.github.com/repos/{REPO}"
root = Path("/content/repo_source")
source_cache = root.parent / "feature_sources"
try:
    token = str(userdata.get("GITHUB_TOKEN") or "").strip()
except Exception:
    token = ""
if not token:
    raise RuntimeError("Repo private: trong Colab → Secrets, thêm GITHUB_TOKEN có quyền Contents: read và Metadata: read cho VietUniverse/ips-wifi-research, bật quyền truy cập cho notebook rồi chạy lại ô này.")
headers = {"Accept": "application/vnd.github+json", "Authorization": f"Bearer {token}", "X-GitHub-Api-Version": "2022-11-28"}
token = None  # Không giữ token trong biến sau khi tạo headers

def github_get(url, timeout=120):
    request = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return response.read()
    except urllib.error.HTTPError as error:
        if error.code == 401:
            raise RuntimeError("GitHub HTTP 401: token sai/hết hạn hoặc thiếu quyền Contents: read + Metadata: read. Tạo token mới, cập nhật Colab Secret GITHUB_TOKEN, rồi chạy lại ô đầu.") from None
        if error.code == 404:
            raise RuntimeError("GitHub HTTP 404: repo/nhánh không truy cập được. Kiểm tra tên repo và quyền của token với repo private.") from None
        raise RuntimeError(f"GitHub API lỗi HTTP {error.code}; kiểm tra mạng hoặc quyền token.") from None
    except urllib.error.URLError as error:
        raise RuntimeError(f"Không kết nối được GitHub: {error.reason}") from None

def extract_branch(branch, destination):
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    archive = github_get(f"{API}/zipball/{branch}")
    with zipfile.ZipFile(io.BytesIO(archive)) as zipped:
        zipped.extractall(destination)
    project_files = list(destination.rglob("pyproject.toml"))
    if len(project_files) != 1:
        raise RuntimeError(f"Nhánh {branch}: cần đúng một pyproject.toml, tìm thấy {len(project_files)}")
    return project_files[0].parent

for folder in (root, source_cache):
    if folder.exists():
        shutil.rmtree(folder)
root.mkdir(parents=True)
source_cache.mkdir(parents=True)
source_ref = input("Nhánh GitHub để nạp code [main]: ").strip() or "main"
project = extract_branch(source_ref, root)
branch_list = json.loads(github_get(f"{API}/branches?per_page=100", timeout=30))
feature_branches = sorted(item["name"] for item in branch_list if item["name"].startswith("feature/"))
if not feature_branches:
    print("GitHub không trả về nhánh feature/*. Sẽ dùng model đã có trong main.")
print(f"Mã nền: {source_ref}")
print("Đang quét các nhánh nhóm:", feature_branches or "(chưa có)")

def is_plugin_or_config(relative):
    parts = relative.parts
    if len(parts) == 4 and parts[:3] == ("src", "ips", "algorithms") and relative.suffix == ".py":
        return True
    if len(parts) == 5 and parts[:4] == ("src", "ips", "data", "preprocessors") and relative.suffix == ".py":
        return True
    if len(parts) == 4 and parts[:3] == ("src", "ips", "neighbors") and relative.suffix == ".py":
        return True
    if len(parts) == 4 and parts[:3] == ("src", "ips", "experiments") and parts[3].startswith("knn_") and relative.suffix == ".py":
        return True
    if len(parts) == 4 and parts[:3] == ("src", "ips", "data") and parts[3] == "dataset_adapters.py":
        return True
    if len(parts) == 2 and parts[0] == "configs" and relative.suffix == ".json":
        return True
    return False

loaded, skipped = {}, []
for branch in feature_branches:
    branch_dir = source_cache / branch.replace("/", "__")
    try:
        branch_project = extract_branch(branch, branch_dir)
    except (RuntimeError, zipfile.BadZipFile) as error:
        print(f"Bỏ qua {branch}: {error}")
        continue
    count = 0
    for source in branch_project.rglob("*"):
        if not source.is_file():
            continue
        relative = source.relative_to(branch_project)
        if not is_plugin_or_config(relative):
            continue
        target = project / relative
        if target.exists():
            skipped.append(f"{branch}:{relative.as_posix()}")
            continue  # main/nhánh đã tải trước là nguồn chuẩn cho file trùng tên
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)
        count += 1
    loaded[branch] = count
    print(f"  {branch}: thêm {count} file plugin/config")

print("Tổng file trùng tên được giữ nguyên theo main/nhánh đầu tiên:", len(skipped))

# Luôn ưu tiên package vừa tải từ main. Colab có thể đã nạp một package tên
# `ips` khác ở lần chạy trước, khiến `ips.data` không được tìm thấy.
import importlib
source_path = str(project / "src")
sys.path[:] = [entry for entry in sys.path if entry != source_path]
sys.path.insert(0, source_path)
for module_name in list(sys.modules):
    if module_name == "ips" or module_name.startswith("ips."):
        del sys.modules[module_name]
importlib.invalidate_caches()

subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(project)], check=True)
try:
    from ips.data.dataset_adapters import PACK_DATASETS
except ModuleNotFoundError as error:
    adapter_path = project / "src" / "ips" / "data" / "dataset_adapters.py"
    raise RuntimeError(
        "Không nạp được ips.data. Hãy kiểm tra notebook đang dùng nhánh main, "
        "sau đó chạy lại ô khởi tạo đầu tiên. "
        f"Package source: {project / 'src'}; adapter tồn tại: {adapter_path.exists()}"
    ) from error
print("Package check OK:", Path(sys.modules["ips"].__file__).resolve())
print("Dataset adapters:", list(PACK_DATASETS))
print("Project:", project, "| Python:", sys.executable)


In [ ]:
# Chọn một hoặc nhiều bộ dữ liệu. Ví dụ: 1,7 để chạy UJIIndoorLoc và DLULIB.
# Dataset ZIP được tìm trong Google Drive trước; chỉ upload nếu chưa có ZIP.

from pathlib import Path
import shutil
import urllib.request
import zipfile
from google.colab import drive, files
from ips.data.dataset_adapters import PACK_DATASETS, prepare_pack_dataset

# Menu dataset
dataset_options = [("ujiindoorloc", "UJIIndoorLoc (UCI) — tải tự động")] + list(PACK_DATASETS.items())
print("Chọn bộ dữ liệu (có thể nhập nhiều số, phân cách bằng dấu phẩy):")
for i, (_, label) in enumerate(dataset_options, 1):
    print(f"  {i}. {label}")

while True:
    dataset_raw = input(f"Nhập số 1..{len(dataset_options)}, ví dụ 1,7: ").strip()
    try:
        selected_indices = [int(value.strip()) for value in dataset_raw.split(",")]
        valid = (
            bool(selected_indices)
            and len(selected_indices) == len(set(selected_indices))
            and all(1 <= value <= len(dataset_options) for value in selected_indices)
        )
        if valid:
            break
    except ValueError:
        pass
    print("Lựa chọn không hợp lệ; nhập các số khác nhau, phân cách bằng dấu phẩy.")

selected_datasets = [
    (dataset_options[i - 1][0], dataset_options[i - 1][1])
    for i in selected_indices
]
data_root = Path("/content/data")
data_root.mkdir(parents=True, exist_ok=True)

# Chỉ cần research ZIP nếu có chọn dataset ngoài UJIIndoorLoc.
pack_zip_path = None
if any(key != "ujiindoorloc" for key, _ in selected_datasets):
    drive.mount("/content/drive")
    zip_dir = Path("/content/drive/MyDrive/ips-data")
    zip_dir.mkdir(parents=True, exist_ok=True)

    # Tìm ZIP đã lưu trên Drive trước. Không upload lại nếu đã có.
    zip_files = sorted(zip_dir.glob("*.zip"))
    if not zip_files:
        print(
            "Chưa có ZIP trong Google Drive/MyDrive/ips-data. "
            "Chọn research ZIP một lần; notebook sẽ lưu lại vào Drive."
        )
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
        if len(zip_names) != 1:
            raise ValueError(
                f"Cần upload đúng một file .zip; nhận được {len(zip_names)} file ZIP."
            )
        uploaded_name = zip_names[0]
        pack_zip_path = zip_dir / Path(uploaded_name).name
        pack_zip_path.write_bytes(uploaded[uploaded_name])
        del uploaded
        zip_files = [pack_zip_path]
        print("Đã lưu ZIP vào Drive:", pack_zip_path)

    # Nếu Drive có nhiều ZIP thì cho chọn file cần dùng.
    if pack_zip_path is None:
        if len(zip_files) == 1:
            pack_zip_path = zip_files[0]
        else:
            print("Chọn research ZIP:")
            for i, path in enumerate(zip_files, 1):
                print(f"  {i}. {path.name}")
            while True:
                zip_raw = input(f"Nhập số 1..{len(zip_files)}: ").strip()
                if zip_raw.isdigit() and 1 <= int(zip_raw) <= len(zip_files):
                    pack_zip_path = zip_files[int(zip_raw) - 1]
                    break
                print("Số không hợp lệ, hãy nhập lại.")

    print("Đọc research ZIP từ Drive:", pack_zip_path)

# Nạp các dataset đã chọn.
dataset_runs = []
for dataset_key, dataset_label in selected_datasets:
    data_dir = data_root / dataset_key
    data_dir.mkdir(parents=True, exist_ok=True)

    if dataset_key == "ujiindoorloc":
        required_files = ("trainingData.csv", "validationData.csv")
        if not all((data_dir / name).exists() for name in required_files):
            zip_path = Path("/content/ujiindoorloc.zip")
            extracted = Path("/content/uji_source")
            extracted.mkdir(parents=True, exist_ok=True)
            print("Đang tải UJIIndoorLoc từ UCI...")
            urllib.request.urlretrieve(
                "https://archive.ics.uci.edu/static/public/310/ujiindoorloc.zip",
                zip_path,
            )
            with zipfile.ZipFile(zip_path) as zipped:
                zipped.extractall(extracted)
            for name in required_files:
                matches = list(extracted.rglob(name))
                if len(matches) != 1:
                    raise RuntimeError(f"Không tìm thấy duy nhất {name} trong ZIP UJI.")
                shutil.copy2(matches[0], data_dir / name)

        data_summary = {
            "dataset": dataset_key,
            "name": dataset_label,
            "n_train": sum(1 for _ in (data_dir / "trainingData.csv").open()) - 1,
            "n_test": sum(1 for _ in (data_dir / "validationData.csv").open()) - 1,
            "n_source_aps": 520,
            "n_features": 520,
        }
    else:
        if pack_zip_path is None or not pack_zip_path.exists():
            raise FileNotFoundError(
                "Không tìm thấy research ZIP. Hãy đặt ZIP trong "
                "Google Drive/MyDrive/ips-data rồi chạy lại cell."
            )
        with pack_zip_path.open("rb") as pack_zip:
            data_summary = prepare_pack_dataset(pack_zip, dataset_key, data_dir)

    dataset_runs.append({
        "key": dataset_key,
        "label": dataset_label,
        "data_dir": data_dir,
        "summary": data_summary,
    })
    print("Dataset:", data_summary)
    print("CSV:", [(p.name, p.stat().st_size) for p in data_dir.glob("*.csv")])

# Giữ các biến tương thích với cell model/training phía dưới.
dataset_key, dataset_label = dataset_runs[0]["key"], dataset_runs[0]["label"]
data_dir = dataset_runs[0]["data_dir"]


## Chọn model và tiền xử lý

Ô sau cho chọn nhiều model và nhiều cách xử lý RSS. Notebook tự ghép preset phù hợp riêng cho từng model; nếu preset dành riêng cho kiểu RSS đó tồn tại thì sẽ ưu tiên preset ấy, nếu không sẽ dùng preset nền của đúng model (hoặc preset KAN nền cho biến thể KAN chưa có file cấu hình riêng). Mỗi cặp model × tiền xử lý chạy độc lập và có thư mục kết quả riêng. KNN/WKNN cũng chạy riêng theo từng phương pháp và từng tiền xử lý. Khi chạy từ hai model neural trở lên, notebook lưu dự đoán validation và tự fit trọng số ensemble theo validation cho từng building; cùng nhóm dataset và RSS mới được ghép. KAN-Mixer preset spatial-aux thêm phân loại cụm tọa độ riêng cho từng building; nhãn cụm chỉ lấy từ tọa độ train, validation chọn epoch và hệ số residual. Official test chỉ dùng để báo cáo sau khi checkpoint được chọn bằng validation nội bộ.


In [ ]:
import json

options = json.loads(subprocess.check_output(
    [sys.executable, "-m", "ips.experiments.cli", "--list-options"], text=True))
presets = sorted(p.relative_to(project).as_posix() for p in (project / "configs").glob("*.json"))
preset_settings = {p: json.loads((project / p).read_text()) for p in presets}
mixer_model_bases = {
    "kan_mixer": "bsrbf_kan",
    "kan_spline_mixer": "kan",
    "fastkan_mixer": "fastkan",
    "skan_mixer": "skan",
    "fckan_mixer": "fckan",
}
model_labels = {
    "kan": "KAN gốc (B-spline)",
    "fastkan": "FastKAN (RBF)",
    "bsrbf_kan": "BSRBF-KAN (B-spline + RBF)",
    "skan": "S-KAN (chọn không gian hàm)",
    "fckan": "FC-KAN (kết hợp hàm)",
    "kan_mixer": "KAN-Mixer (BSRBF-KAN + Residual Mixer)",
    "kan_spline_mixer": "KAN-Mixer (KAN gốc / B-spline)",
    "fastkan_mixer": "KAN-Mixer (FastKAN)",
    "skan_mixer": "KAN-Mixer (S-KAN)",
    "fckan_mixer": "KAN-Mixer (FC-KAN)",
}
model_ids = {label: model_id for model_id, label in model_labels.items()}
neural_ids = sorted(set(options["models"] + list(model_labels)))
families = {"Neural": [model_labels.get(model_id, model_id) for model_id in neural_ids]}
if (project / "src/ips/neighbors/__init__.py").is_file():
    neighbor_probe = subprocess.run(
        [sys.executable, "-c", "import json; from ips.neighbors import available_models; print(json.dumps(sorted(available_models())))"],
        capture_output=True, text=True)
    if neighbor_probe.returncode:
        raise RuntimeError(f"Không tải được plugin KNN: {neighbor_probe.stderr}")
    families["KNN/WKNN"] = json.loads(neighbor_probe.stdout)

def choose_one(title, values):
    if not values:
        raise ValueError(f"Không có lựa chọn cho {title}")
    print("\n" + title)
    for i, value in enumerate(values, 1):
        print(f"  {i}. {value}")
    while True:
        raw = input(f"Nhập số 1..{len(values)}: ").strip()
        if raw.isdigit() and 1 <= int(raw) <= len(values):
            return values[int(raw) - 1]
        print("Số không hợp lệ, hãy nhập lại.")

def choose_many(title, values):
    if not values:
        raise ValueError(f"Không có lựa chọn cho {title}")
    print("\n" + title)
    for i, value in enumerate(values, 1):
        print(f"  {i}. {value}")
    while True:
        raw = input("Nhập nhiều số cách nhau bằng dấu phẩy, ví dụ 1,3: ").strip()
        try:
            indices = [int(x.strip()) for x in raw.split(",")]
            if indices and len(indices) == len(set(indices)) and all(1 <= i <= len(values) for i in indices):
                return [values[i - 1] for i in indices]
        except ValueError:
            pass
        print("Danh sách không hợp lệ, hãy nhập lại.")

selected_family = choose_one("Chọn họ model", list(families))
selected_choices = choose_many("Chọn model", families[selected_family])
selected_models = [model_ids.get(choice, choice) for choice in selected_choices]
preprocess_options = options["preprocessors"]
selected_preprocesses = choose_many("Chọn một hoặc nhiều cách xử lý RSS", preprocess_options)

mixer_variants = []
if any(name in mixer_model_bases for name in selected_models):
    variant_options = [
        ("kan", "kan", "KAN ở token + KAN ở channel"),
        ("kan", "mlp", "KAN ở token + MLP ở channel"),
        ("mlp", "kan", "MLP ở token + KAN ở channel"),
        ("mlp", "mlp", "MLP ở token + MLP ở channel"),
    ]
    print("\nChọn cấu trúc Mixer (KAN dùng đúng họ model đã chọn):")
    for i, (_, _, label) in enumerate(variant_options, 1):
        print(f"  {i}. {label}")
    selected_variant_indices = choose_many(
        "Chọn một hoặc nhiều cấu trúc Mixer", [v[2] for v in variant_options])
    mixer_variants = [v for v in variant_options if v[2] in selected_variant_indices]

spatial_aux_enabled = False
if "kan_mixer" in selected_models:
    spatial_aux_enabled = choose_one(
        "Học thêm cụm vị trí riêng cho từng building?",
        ["Không (KAN-Mixer chuẩn)", "Có (KAN-Mixer + spatial cluster auxiliary, 8 cụm)"],
    ).startswith("Có")

def candidates_for_model(model_name):
    if selected_family == "KNN/WKNN":
        return [p for p in presets if Path(p).name.startswith("knn_")]
    run_model = mixer_model_bases.get(model_name, model_name)
    wants_mixer = model_name in mixer_model_bases
    direct = [p for p, cfg in preset_settings.items()
              if cfg.get("model") == run_model
              and bool(cfg.get("with_mixer", False)) == wants_mixer]
    if direct:
        return direct
    # FastKAN/FC-KAN/S-KAN configs may share the common KAN training template.
    if run_model in {"kan", "fastkan", "fckan", "skan"}:
        return [p for p, cfg in preset_settings.items()
                if cfg.get("model") == "kan"
                and bool(cfg.get("with_mixer", False)) == wants_mixer]
    return []

def choose_auto_preset(model_name, preprocess_name):
    if (spatial_aux_enabled and model_name == "kan_mixer"
            and "configs/bsrbf_mixer_spatial_aux.json" in preset_settings):
        return "configs/bsrbf_mixer_spatial_aux.json"
    candidates = candidates_for_model(model_name)
    if not candidates:
        raise ValueError(f"Không tìm được preset nền phù hợp cho model {model_name}.")
    # Exact preprocessing preset first, then this model's baseline/full config,
    # with smoke-test configs used only as a last resort.
    def score(path):
        cfg = preset_settings[path]
        return (cfg.get("preprocess") == preprocess_name,
                cfg.get("mixer_arch") == "ap_token",
                "smoke" not in Path(path).name.lower(),
                "baseline" in Path(path).name.lower(),
                path)
    selected = max(candidates, key=score)
    return selected

selected_preset_map = {}
for model_name in selected_models:
    for preprocess_name in selected_preprocesses:
        selected_preset_map[(model_name, preprocess_name)] = choose_auto_preset(
            model_name, preprocess_name)

preprocess_params_by_name = {}
for preprocess_name in selected_preprocesses:
    defaults = {}
    matching = [preset_settings[p] for (model_name, prep), p in selected_preset_map.items()
                if prep == preprocess_name
                and preset_settings[p].get("preprocess") == preprocess_name]
    if matching:
        defaults = matching[0].get("preprocess_params", {})
    raw = input(f"Tham số cho {preprocess_name} (Enter để dùng {json.dumps(defaults, ensure_ascii=False)}): ").strip()
    params = json.loads(raw or json.dumps(defaults))
    if not isinstance(params, dict):
        raise ValueError("RSS params phải là JSON object.")
    preprocess_params_by_name[preprocess_name] = params

print("\nKế hoạch chạy:")
for (model_name, preprocess_name), preset in selected_preset_map.items():
    template_model = preset_settings[preset].get("model", "KNN/WKNN")
    note = "" if template_model == model_name else f" (preset nền {template_model}; model sẽ được override thành {model_name})"
    print(f"  {model_labels.get(model_name, model_name)} × {preprocess_name} → {preset}{note}")
if mixer_variants:
    print("Biến thể Mixer:", ", ".join(v[2] for v in mixer_variants))
print("Tham số RSS:", json.dumps(preprocess_params_by_name, ensure_ascii=False, indent=2))


In [ ]:
import torch
from datetime import datetime, timezone

RUN_NAME = datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S")
run_dir = Path("/content/runs") / RUN_NAME
from ips.experiments.visualize import render_run, compare_runs

def export_floor_metrics(output_dir, data_dir):
    """Report positioning error grouped by the ground-truth floor."""
    import pandas as pd

    output_dir = Path(output_dir)
    floor_truth = pd.read_csv(data_dir / "validationData.csv", usecols=["FLOOR"])
    records = []
    for prediction_path in sorted(output_dir.rglob("predictions.csv")):
        pred = pd.read_csv(prediction_path)
        if "true_floor" in pred:
            floors = pred["true_floor"].to_numpy()
        elif "source_row" in pred:
            rows = pred["source_row"].astype(int).to_numpy()
            if len(rows) and (rows.min() < 0 or rows.max() >= len(floor_truth)):
                raise IndexError(f"source_row ngoài validationData.csv: {prediction_path}")
            floors = floor_truth.iloc[rows]["FLOOR"].to_numpy()
        else:
            print(f"Bỏ qua predictions không có true_floor/source_row: {prediction_path}")
            continue

        error_columns = (["error_m"] if "error_m" in pred.columns else
                         [c for c in pred.columns if c.endswith("_error_m")])
        if not error_columns:
            print(f"Bỏ qua predictions không có cột lỗi vị trí: {prediction_path}")
            continue
        for column in error_columns:
            model_name = (column[:-len("_error_m")] if column.endswith("_error_m")
                          else output_dir.name)
            errors = pred[column].to_numpy(dtype=float)
            records.extend({"model": model_name, "floor": floor, "error_m": error}
                           for floor, error in zip(floors, errors))

    if not records:
        print("Không tìm thấy predictions phù hợp để tính sai số từng tầng.")
        return None

    samples = pd.DataFrame(records)
    report = (samples.groupby(["model", "floor"])["error_m"]
              .agg(n_test="count", mean_m="mean", median_m="median",
                   p90_m=lambda x: x.quantile(0.90),
                   within_5m_pct=lambda x: 100 * (x <= 5).mean())
              .reset_index().sort_values(["model", "floor"]))
    report_path = output_dir / "floor_metrics.csv"
    report.to_csv(report_path, index=False)
    print("\nSai số định vị theo tầng (tầng thật):")
    display(report.round(3))
    print("Đã lưu:", report_path)
    return report

def compare_dataset_runs(run_records, output_dir):
    """Compare aggregate localization metrics across datasets (different test sets)."""
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd

    rows = []
    for record in run_records:
        model_dir = Path(record["output_dir"])
        prediction_paths = sorted(model_dir.glob("building_*/predictions.csv"))
        errors_by_model = {}
        for prediction_path in prediction_paths:
            pred = pd.read_csv(prediction_path)
            error_columns = (["error_m"] if "error_m" in pred.columns else
                             [c for c in pred.columns if c.endswith("_error_m")])
            for column in error_columns:
                model_name = record["model_label"]
                errors_by_model.setdefault(model_name, []).append(
                    pred[column].to_numpy(dtype=float))
        for model_name, parts in errors_by_model.items():
            errors = np.concatenate(parts)
            rows.append({
                "dataset": record["dataset_label"], "model": model_name,
                "n_test": len(errors), "mean_m": errors.mean(),
                "median_m": np.median(errors), "p90_m": np.quantile(errors, .9),
                "within_5m_pct": 100 * np.mean(errors <= 5),
            })

    if not rows:
        print("Không có predictions để so sánh các bộ dữ liệu.")
        return None
    table = pd.DataFrame(rows).sort_values(["dataset", "model"])
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    csv_path = output_dir / "dataset_comparison_metrics.csv"
    table.to_csv(csv_path, index=False)

    labels = [f"{row.dataset}\n{row.model}" for row in table.itertuples()]
    fig, ax = plt.subplots(figsize=(max(8, len(table) * 1.5), 5))
    bars = ax.bar(labels, table["mean_m"], color=plt.cm.Set2(np.linspace(0, 1, len(table))))
    ax.bar_label(bars, fmt="%.2f m", padding=3, fontsize=8)
    ax.set(title="Positioning error across datasets", ylabel="Mean Euclidean error (m)")
    ax.grid(axis="y", alpha=.25)
    ax.set_axisbelow(True)
    fig.tight_layout()
    mean_path = output_dir / "dataset_comparison_mean_error.png"
    fig.savefig(mean_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 5))
    for record in run_records:
        model_dir = Path(record["output_dir"])
        errors_by_model = {}
        for prediction_path in sorted(model_dir.glob("building_*/predictions.csv")):
            pred = pd.read_csv(prediction_path)
            error_columns = (["error_m"] if "error_m" in pred.columns else
                             [c for c in pred.columns if c.endswith("_error_m")])
            for column in error_columns:
                model_name = record["model_label"]
                errors_by_model.setdefault(model_name, []).append(pred[column].to_numpy(float))
        for model_name, parts in errors_by_model.items():
            errors = np.sort(np.concatenate(parts))
            ax.step(errors, np.arange(1, len(errors) + 1) / len(errors), where="post",
                    label=f"{record['dataset_label']}: {model_name}")
    ax.set(xlabel="Positioning error (m)", ylabel="Fraction of test samples",
           ylim=(0, 1.02), title="Positioning error CDF across datasets")
    ax.grid(alpha=.25)
    ax.legend(fontsize=8)
    fig.tight_layout()
    cdf_path = output_dir / "dataset_comparison_cdf.png"
    fig.savefig(cdf_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    print("\nSo sánh tổng hợp theo bộ dữ liệu (test split riêng, không ghép cặp từng mẫu):")
    print("Hãy xác nhận tọa độ các bộ dữ liệu đều tính bằng mét trước khi diễn giải mean error.")
    display(table.round(3))
    print("Đã lưu:", csv_path, mean_path, cdf_path)
    return table

run_dirs = []
run_labels = []
dataset_run_records = []
if not selected_models or not selected_preprocesses:
    raise ValueError("Hãy chọn ít nhất một model và một cách xử lý RSS.")
print("GPU available:", torch.cuda.is_available())
if selected_family == "Neural" and any(
        mixer_model_bases.get(name, name) in {"bsrbf_kan", "kan", "fastkan", "skan", "fckan"}
        for name in selected_models) and not torch.cuda.is_available():
    print("Cảnh báo: các biến thể KAN có thể huấn luyện chậm trên CPU.")
print("Số dòng dữ liệu:", [(p.name, sum(1 for _ in p.open()) - 1)
                           for p in dataset_runs[0]["data_dir"].glob("*.csv")])

for dataset in dataset_runs:
    dataset_key, dataset_label, data_dir = dataset["key"], dataset["label"], dataset["data_dir"]
    print("\nDataset:", dataset_label, "| train/test:",
          dataset["summary"]["n_train"], dataset["summary"]["n_test"])
    for selected_model in selected_models:
        for selected_preprocess in selected_preprocesses:
            selected_preset = selected_preset_map[(selected_model, selected_preprocess)]
            base_settings = dict(preset_settings[selected_preset])
            base_settings["preprocess"] = selected_preprocess
            base_settings["preprocess_params"] = preprocess_params_by_name[selected_preprocess]
            if dataset_key != "ujiindoorloc":
                base_settings["buildings"] = [0]
            model_label = model_labels.get(selected_model, selected_model)
            variants = (mixer_variants if selected_model in mixer_model_bases
                        else [(None, None, "")])

            for token_type, channel_type, variant_label in variants:
                settings = dict(base_settings)
                display_name = f"{model_label} [{selected_preprocess}]"
                variant_slug = ""
                if variant_label:
                    display_name += f" [token={token_type.upper()}, channel={channel_type.upper()}]"
                    variant_slug = f"__token_{token_type}_channel_{channel_type}"
                    settings.update(mixer_arch="ap_token", mixer_token_type=token_type,
                                    mixer_channel_type=channel_type, mixer_isolate_rng=True)
                if spatial_aux_enabled and selected_model == "kan_mixer":
                    if dataset_key == "ujiindoorloc":
                        settings.update(mixer_arch="ap_token", spatial_clusters=8,
                                        spatial_aux_weight=0.1)
                    else:
                        settings["spatial_clusters"] = 0

                model_dir = run_dir / dataset_key / selected_preprocess / f"{selected_model}{variant_slug}"
                selected_config = Path("/content") / (
                    f"selected_{RUN_NAME}_{dataset_key}_{selected_model}_"
                    f"{selected_preprocess}{variant_slug}.json")
                if selected_family == "KNN/WKNN":
                    settings["models"] = [selected_model]
                    command = [sys.executable, "-u", "-m", "ips.experiments.knn_cli",
                               "--config", str(selected_config), "--data-dir", str(data_dir),
                               "--output-dir", str(model_dir)]
                else:
                    run_model = mixer_model_bases.get(selected_model, selected_model)
                    settings["model"] = run_model
                    settings["with_mixer"] = selected_model in mixer_model_bases
                    command = [sys.executable, "-u", "-m", "ips.experiments.cli",
                               "--config", str(selected_config), "--model", run_model,
                               "--preprocess", selected_preprocess,
                               "--preprocess-params", json.dumps(preprocess_params_by_name[selected_preprocess]),
                               "--data-dir", str(data_dir), "--output-dir", str(model_dir)]

                selected_config.write_text(json.dumps(settings, ensure_ascii=False), encoding="utf-8")
                print("\nĐang chạy:", display_name, "| preset:", selected_preset)
                print("Cấu hình:", json.dumps(settings, ensure_ascii=False, indent=2))
                subprocess.run(command, check=True)
                export_floor_metrics(model_dir, data_dir)
                render_run(model_dir)
                run_dirs.append(model_dir)
                run_labels.append(display_name)
                dataset_run_records.append({"dataset_key": dataset_key,
                                            "dataset_label": dataset_label,
                                            "output_dir": model_dir,
                                            "model_label": display_name,
                                            "preprocess_label": selected_preprocess})

if selected_family == "Neural":
    from ips.experiments.ensemble import validation_weighted_ensemble
    validation_weighted_ensemble(dataset_run_records, run_dir / "validation_ensemble")

if len(dataset_runs) > 1:
    compare_dataset_runs(dataset_run_records, run_dir)
elif len(run_dirs) > 1:
    compare_runs(run_dirs, labels=run_labels)
print("Kết quả:", run_dir)


In [ ]:
archive = shutil.make_archive(str(run_dir), "zip", root_dir=run_dir)
print("Result ZIP:", archive)
files.download(archive)
